# The Critical Line — a guided walkthrough

**This notebook does not prove the Riemann Hypothesis.** RH is open, nothing
here bears on it, and every number below is finite-precision numerical
exploration — experimental evidence, never a certified result.

What it does is walk through the project's main computations in order, so you
can see what each one actually establishes.

Run `pip install -e .` from the repository root first.

## 0. Setup

In [ ]:
import mpmath as mp

from critical_line import counting, explicit_formula, gram, hardy_z, spacing, zeros, zeta_tools
from critical_line.constants import COUNTING_COMPARISON_NOTE, DISCLAIMER_LONG, LEHMER_NOTE

print(DISCLAIMER_LONG)

## 1. The functional equation

Riemann's functional equation relates `zeta(s)` to `zeta(1-s)`. It is a theorem;
checking it numerically tests our implementation, not the mathematics.

Note the `mp.workdps` context around the comparison. Without it, the subtraction
happens at the ambient 15 digits and the residual cannot fall below ~1e-16 no
matter how accurate the two sides are — a trap documented in
`docs/numerical_methods.md`.

In [ ]:
with mp.workdps(30):
    for s in [mp.mpc('0.3', '1.7'), mp.mpc('-0.5', '4.0'), mp.mpc('0.75', '30.0')]:
        r = zeta_tools.functional_equation_residual(s, dps=30)
        print(f'{str(s):>22}   |lhs - rhs| = {mp.nstr(r, 5)}')

And the completed function `xi(s) = xi(1-s)`. The symmetry is about the line
`Re(s) = 1/2` — which is why that line is the natural axis of the problem.

In [ ]:
with mp.workdps(30):
    s = mp.mpc('0.3', '1.7')
    print('xi(s)      =', mp.nstr(zeta_tools.xi(s, dps=30), 15))
    print('xi(1-s)    =', mp.nstr(zeta_tools.xi(1 - s, dps=30), 15))
    print('difference =', mp.nstr(abs(zeta_tools.xi(s, dps=30) - zeta_tools.xi(1 - s, dps=30)), 5))

## 2. Hardy's Z function

`Z(t) = exp(i*theta(t)) * zeta(1/2 + i*t)` is real for real `t`, and vanishes
exactly where `zeta(1/2 + it)` does. So sign changes of `Z` bracket zeros
**on the critical line**.

`Z` cannot see zeros off the line at all. Keep that in mind — it is the whole
reason the next section exists.

Two entry points for the same quantity. Note what mpmath actually does:
`mp.siegelz` only uses the Riemann-Siegel asymptotic expansion when
`t > 500 * prec` (about `t > 51500` at 30 digits). Below that -- which is
everywhere in this notebook -- it computes `expj(siegeltheta(t)) * zeta(1/2+it)`,
the same formula as `hardy_z_from_zeta`. So their agreement below that height
confirms our wiring, not two independent evaluations of Z.

In [ ]:
for t in [14.5, 25.0, 50.5]:
    a = hardy_z.hardy_z(t, dps=30)
    b = hardy_z.hardy_z_from_zeta(t, dps=30)
    imag = hardy_z.hardy_z_imaginary_residual(t, dps=30)
    print(f't = {t:>6}   Z = {float(a):+.12f}   |diff| = {mp.nstr(abs(a - b), 4):>10}'
          f'   discarded imag = {mp.nstr(imag, 4)}')

The discarded imaginary part is mathematically zero; what you see is a finite-precision residual.

## 3. Finding zeros on the critical line

Scan for sign changes, then bisect. This does **not** call `mp.zetazero`, so it
can be compared against mpmath's Rosser-block machinery as a genuine second
opinion.

In [ ]:
gammas = zeros.find_on_line_zeros(60, dps=25)
print(f'{len(gammas)} zeros below T = 60\n')
print(f"{'k':>3}  {'ours (grid + bisection)':>26}  {'mp.zetazero':>26}  {'diff':>10}")
for k, g in enumerate(gammas, start=1):
    want = mp.zetazero(k).imag
    print(f'{k:>3}  {mp.nstr(g, 18):>26}  {mp.nstr(want, 18):>26}  {mp.nstr(abs(g - want), 3):>10}')

## 4. The circularity trap

Before the real comparison, the mistake it replaced.

`mp.nzeros(T)` looks like an independent count of zeros. It is not: mpmath
builds it from `gram_index = floor(theta(T)/pi)` plus sign-change bookkeeping.
Since `S(T)` is *defined* as `N(T) - theta(T)/pi - 1`, comparing `nzeros`
against the Riemann-von Mangoldt formula just recovers `S(T)`.

Watch the residual column.

In [ ]:
with mp.workdps(30):
    print(f"{'T':>6}  {'nzeros(T)':>10}  {'theta/pi+1':>12}  {'S(T)':>10}  {'residual':>12}")
    for T in [50, 100, 200, 500, 1000]:
        n = mp.nzeros(T)
        main = counting.rvm_main_term(T, dps=30)
        s_val = counting.S(T, dps=30)
        print(f'{T:>6}  {n:>10}  {float(main):>12.4f}  {float(s_val):>+10.4f}'
              f'  {mp.nstr(abs(mp.mpf(n) - main - s_val), 3):>12}')

Zero to working precision. One identity rearranged — it says nothing about where any zero is.

## 5. The comparison that does carry information

Two counts reached by genuinely different routes:

- **strip count** — a contour integral of `zeta'/zeta` around a rectangle
  enclosing the whole critical strip. Counts every zero inside, *wherever it
  sits*. The integrand mentions only zeta and its derivative.
- **on-line count** — sign changes of `Z`.

If they agree, every zero below `T` is on the critical line and simple.

These use different numerical counting procedures, but both rely on
finite-precision numerical evaluation and therefore provide experimental
evidence rather than a certified result. They are not fully independent:
every value either one needs comes out of `mp.zeta`.

In [ ]:
record = counting.experimentally_check_up_to(50, dps=25, contour_dps=30, maxdegree=8, verbose=True)
print()
print(record['note'])

Note the function is called `experimentally_check_up_to`, not `certify_up_to`.
`mpmath` is not interval-certified, so "certify" would misdescribe the output.

Also note `count_zeros_argument_principle_int` refuses to round a result that
is not close enough to an integer, rather than returning a confident wrong
answer:

In [ ]:
try:
    counting.count_zeros_argument_principle_int(
        20, maxdegree=4, dps=20, residual_tolerance=mp.mpf('1e-300')
    )
except counting.ResidualToleranceError as e:
    print(type(e).__name__)
    print()
    print(e)

## 6. Gram points and Gram's law

`theta(g_n) = n*pi`. Gram's law says `Z` alternates in sign at consecutive Gram
points. It is an empirical tendency, known to fail infinitely often, first at
`n = 126` — **not a proof technique**.

In [ ]:
rows = gram.gram_interval_report(0, 10, dps=25)
print(f"{'n':>3}  {'g_n':>14}  {'Z(g_n)':>14}  {'expected':>9}  {'observed':>9}  {'obeys':>6}")
for r in rows:
    print(f"{r['n']:>3}  {float(r['g_n']):>14.6f}  {float(r['z_at_g_n']):>+14.6f}"
          f"  {r['expected_sign']:>9}  {r['observed_sign']:>9}  {str(r['obeys_gram_law']):>6}")

print('\nfailures for n in [1, 40):', gram.gram_law_failures(1, 40, dps=25) or 'none')

## 7. Spacing statistics

Gaps between consecutive zeros, unfolded by the local mean spacing
`2*pi / log(gamma / 2*pi)` so the mean is 1 **by construction** — that mean is
not a check on anything. What carries information is the distribution's shape.

In [ ]:
many = zeros.first_n_zeros(150, dps=20)
summary = spacing.spacing_summary(many)
for key, value in summary.items():
    print(f'{key:<20}: {float(value) if not isinstance(value, int) else value}')

print()
print(f'Lehmer-like pairs: {len(spacing.lehmer_like_pairs(many))}')
print(LEHMER_NOTE)

## 8. The explicit formula

Von Mangoldt's formula rebuilds `psi(x)` — a sum over prime powers — from the
zeros of zeta. Add zero pairs one at a time and watch it close in.

The zeros used are placed on the critical line by construction, so this tests
the explicit formula, not RH.

In [ ]:
x = 100.0
truth = explicit_formula.psi(x)
sums = explicit_formula.explicit_formula_partial_sums(x, many, dps=25)

print(f'psi({x:g}) from the primes = {truth:.6f}\n')
print(f"{'zero pairs':>11}  {'partial sum':>14}  {'|error|':>10}")
for k in [0, 1, 5, 10, 25, 50, 100, len(many)]:
    if k < len(sums):
        print(f'{k:>11}  {float(sums[k]):>14.6f}  {float(abs(sums[k] - truth)):>10.6f}')

Convergence is slow and oscillatory — the zeros encode psi's jumps, so partial sums swing around the true value rather than settling smoothly.

## 9. Figures

Each figure carries a visible disclaimer, because figures get separated from
their captions.

In [ ]:
from pathlib import Path

from critical_line import plots

out = Path('../outputs')
print(plots.plot_hardy_z(10, 40, out / 'nb_hardy_z.png', dps=25))
print(plots.plot_zero_ordinates(many, out / 'nb_ordinates.png'))
print(plots.plot_normalized_gap_histogram(many, out / 'nb_gaps.png'))
print(plots.plot_explicit_formula_error(x, many, out / 'nb_explicit.png'))

## Where this stops

Every zero checked above is on the critical line. That is a statement about
finitely many zeros below a finite height, obtained from finite-precision
arithmetic with no rigorous error bounds.

RH is a statement about infinitely many zeros. No computation reaches them, and
the gap is not one that a bigger `T` narrows.

See `docs/mathematical_scope.md` for why, and `docs/certification.md` for what
a genuinely certified finite-height result would require.